# TripMe - Regenerate Place Descriptions (50-80 words, LLM)

Reads every place record already collected under `data/raw/<District>/<category>.jsonl`,
rewrites each `description` field to a fresh 50-80 word LLM-generated description
(Qwen2.5-3B-Instruct), and writes the updated records back out in the same
per-district/per-category JSONL layout - ready to copy back into `data/raw` locally.

## How to run this on Kaggle
1. Zip your local `data/raw` folder and upload it as a Kaggle dataset (e.g. named
   `tripme-raw-places`), then attach it to this notebook via "Add Input".
2. Set `RAW_INPUT_DIR` below to wherever Kaggle mounts it
   (usually `/kaggle/input/<dataset-name>/raw` or `/kaggle/input/<dataset-name>`
   depending on how the zip was structured - check the Kaggle "Input" file browser).
3. Turn on a GPU accelerator (Settings -> Accelerator -> GPU T4 x2 or better).
4. Run all cells. Output files land in `/kaggle/working/raw_updated/` - download
   that folder (Kaggle auto-zips the working directory) and copy its contents
   over `data/raw` locally.

This notebook is resumable: progress is checkpointed to
`/kaggle/working/desc_progress.json`, so if the Kaggle session gets interrupted
partway through, re-running continues where it left off instead of starting over.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece tqdm
print("Dependencies installed.")

In [ ]:
import json
import re
from pathlib import Path

from tqdm.auto import tqdm

# Where the uploaded data/raw folder is mounted. Adjust this to match your
# Kaggle dataset's actual path (check the "Input" panel on the right).
RAW_INPUT_DIR = Path("/kaggle/input/tripme-raw-places/raw")
if not RAW_INPUT_DIR.exists():
    # Fallback for local/non-Kaggle runs (e.g. testing on your own machine).
    RAW_INPUT_DIR = Path("data/raw")

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
RAW_OUTPUT_DIR = OUTPUT_DIR / "raw_updated"
PROGRESS_FILE = OUTPUT_DIR / "desc_progress.json"

RAW_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Reading from:", RAW_INPUT_DIR)
print("Writing to:", RAW_OUTPUT_DIR)

## Load every place record from data/raw

In [ ]:
def load_records(path: Path):
    text = path.read_text(encoding="utf-8")
    decoder = json.JSONDecoder()
    idx = 0
    n = len(text)
    while idx < n:
        while idx < n and text[idx] in " \t\r\n":
            idx += 1
        if idx >= n:
            break
        obj, end = decoder.raw_decode(text, idx)
        yield obj
        idx = end


all_files = sorted(RAW_INPUT_DIR.rglob("*.jsonl"))
print(f"Found {len(all_files)} .jsonl files under {RAW_INPUT_DIR}")

records_by_file = {}
total_records = 0
for f in all_files:
    recs = list(load_records(f))
    records_by_file[f] = recs
    total_records += len(recs)

print(f"Loaded {total_records} place records total.")

## Load the LLM (Qwen2.5-3B-Instruct, 4-bit)

In [ ]:
LLM_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

llm_ready = False
llm_tokenizer = None
llm_model = None

try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    if torch.cuda.is_available():
        print(f"Loading {LLM_MODEL_NAME} (4-bit)...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
        llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
        llm_model = AutoModelForCausalLM.from_pretrained(
            LLM_MODEL_NAME,
            quantization_config=bnb_config,
            device_map="auto",
        )
        llm_model.eval()
        llm_ready = True
        print("LLM loaded and ready.")
    else:
        print("No GPU detected. This notebook needs a GPU to generate real "
              "50-80 word descriptions - enable one via Settings -> Accelerator "
              "and re-run. Continuing would only produce short template text, "
              "which defeats the point of this run, so stopping here.")
        raise SystemExit("No GPU available.")
except ImportError as e:
    raise SystemExit(f"Missing dependency ({e}) - re-run the pip install cell above.")

## Description generation (50-80 words, grounded only in known facts)

In [ ]:
CATEGORY_BLURB = {
    "Buddhist Temple": "a Buddhist temple", "Kovil": "a Hindu kovil",
    "Church": "a church", "Mosque": "a mosque", "Temple": "a place of worship",
    "Ruins": "an archaeological site", "Building": "a historic landmark",
    "Cascade": "a waterfall", "Plunge": "a waterfall", "Tiered": "a waterfall",
    "Fan": "a waterfall", "Horsetail": "a waterfall", "Block": "a waterfall",
    "Segmented": "a waterfall", "Multi-step": "a waterfall",
    "Sandy Beach": "a beach", "Surf Beach": "a beach", "Urban Beach": "a beach",
    "Cove Beach": "a beach", "National Park": "a nature reserve",
    "Viewpoint": "a scenic viewpoint", "Museum": "a museum",
    "Adventure Park": "an adventure/activity park", "Tea Estate": "a tea estate",
    "Devalaya": "a Hindu-Buddhist shrine (devalaya)", "Devale": "a Hindu-Buddhist shrine (devale)",
    "Stupa": "a Buddhist stupa", "Fort": "a historic fort",
    "Other": "a point of interest",
}


def blurb_for(category_id: str) -> str:
    return CATEGORY_BLURB.get(category_id, "a point of interest")


DESC_SYSTEM_PROMPT = (
    "You write natural-sounding descriptions of Sri Lankan tourist places for "
    "a travel app. Use ONLY the facts given to you (name, category, district, "
    "and any activities listed) - never invent history, dates, prices, opening "
    "hours, or other specific facts you were not given. Write exactly one "
    "paragraph, 50 to 80 words long. Do not use markdown, headings, or bullet "
    "points. Reply with only the description text, no preamble or word count."
)


def build_desc_prompt(rec: dict) -> str:
    name = rec.get("name", "")
    category_id = rec.get("category_id", "Other")
    district = rec.get("district_id", "")
    activities = rec.get("activities", "")
    blurb = blurb_for(category_id)
    lines = [
        f'Place name: "{name}"',
        f"Category: {category_id} ({blurb})",
        f"District: {district}, Sri Lanka",
    ]
    if activities:
        lines.append(f"Known activities: {activities}")
    lines.append("\nWrite a 50-80 word description of this place for a travel app.")
    return "\n".join(lines)


def llm_generate(rec: dict, max_new_tokens=160, temperature=0.6) -> str:
    messages = [
        {"role": "system", "content": DESC_SYSTEM_PROMPT},
        {"role": "user", "content": build_desc_prompt(rec)},
    ]
    inputs = llm_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(llm_model.device)
    with torch.no_grad():
        out = llm_model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=llm_tokenizer.eos_token_id,
        )
    text = llm_tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
    return text.strip()


def is_valid_description(text: str, name: str) -> bool:
    if not text:
        return False
    word_count = len(text.split())
    if word_count < 45 or word_count > 90:
        return False
    first_word = re.sub(r"[^a-zA-Z]", "", name.split()[0]) if name.split() else ""
    if first_word and len(first_word) > 2 and first_word.lower() not in text.lower():
        return False
    return True


def generate_description(rec: dict, retries=3) -> str | None:
    for _ in range(retries + 1):
        try:
            text = llm_generate(rec)
        except Exception as e:
            print(f"  generation error for {rec.get('name')!r}: {e}")
            continue
        if is_valid_description(text, rec.get("name", "")):
            return text
    return None  # caller decides what to do if every attempt failed


# Quick smoke test
sample = {"name": "Aluvihara Rock Temple", "category_id": "Buddhist Temple",
          "district_id": "Matale", "activities": "meditation, photography"}
print(generate_description(sample))

## Process every record (resumable via a checkpoint keyed by place id)

In [ ]:
def load_progress() -> dict:
    if PROGRESS_FILE.exists():
        return json.loads(PROGRESS_FILE.read_text(encoding="utf-8"))
    return {"done": {}}  # id -> new description text


def save_progress(progress: dict) -> None:
    PROGRESS_FILE.write_text(json.dumps(progress), encoding="utf-8")


progress = load_progress()
done_map = progress["done"]
print(f"Resuming: {len(done_map)} descriptions already generated in a previous run.")

all_records = [rec for recs in records_by_file.values() for rec in recs]
pending = [rec for rec in all_records if rec.get("id") not in done_map]
print(f"{len(pending)} of {len(all_records)} records need a new description.")

SAVE_EVERY = 25
for i, rec in enumerate(tqdm(pending, desc="Generating descriptions")):
    rid = rec.get("id")
    new_desc = generate_description(rec)
    if new_desc is not None:
        done_map[rid] = new_desc
    # else: leave it out of done_map - the old description is kept as a
    # fallback when writing output below, and it'll be retried on next run.
    if (i + 1) % SAVE_EVERY == 0:
        save_progress(progress)

save_progress(progress)
print(f"\nDone. {len(done_map)} descriptions generated/updated so far "
      f"({len(all_records) - len(done_map)} still pending - re-run this cell to retry them).")

## Write updated records back out, same per-district/per-category layout

In [ ]:
updated_count = 0
kept_old_count = 0

for src_path, recs in records_by_file.items():
    rel = src_path.relative_to(RAW_INPUT_DIR)
    out_path = RAW_OUTPUT_DIR / rel
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        for rec in recs:
            rid = rec.get("id")
            if rid in done_map:
                rec = dict(rec)
                rec["description"] = done_map[rid]
                updated_count += 1
            else:
                kept_old_count += 1
            f.write(json.dumps(rec, indent=4, ensure_ascii=False))
            f.write("\n\n")

print(f"Wrote {len(records_by_file)} files to {RAW_OUTPUT_DIR}")
print(f"{updated_count} records got a new 50-80 word description.")
print(f"{kept_old_count} records kept their old description (generation failed every "
      f"retry, or run was interrupted before reaching them - re-run to fill these in).")
print("\nNext step: download /kaggle/working (Kaggle auto-zips it), then copy the "
      "contents of raw_updated/ over your local data/raw folder.")